# RNA Sequence Features Exploration with `mlcroissant`
This notebook guides you through the exploration of the "RNA Sequence Features from Microbial Genome and Metagenome Assemblies via ONT and Illumina Platforms" dataset, using the `mlcroissant` library.

### Dataset Source
Dataset schema URL:
`https://sen.science/doi/10.71728/senscience.2nmx-acf1/fair2.json`


In [ ]:
# Install mlcroissant if necessary
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.2nmx-acf1/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata using `.to_json()`
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Version: {metadata['version']}")
print(f"Date Published: {metadata['datePublished']}")
print(f"Identifier: {metadata['identifier']}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references are based on the `@id` values from the metadata schema.

In [ ]:
# Examine the top-level record sets with their @id
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']} | name: {rs.get('name', '[No name]')} | description: {rs.get('description', '[No description]')}")
        if 'fields' in rs and rs['fields']:
            print("  Fields:")
            for field in rs['fields']:
                f_id = field['@id']
                f_name = field.get('name', '[No name]')
                f_type = field.get('dataType', '[No dataType]')
                print(f"    - @id: {f_id}, name: {f_name}, dataType: {f_type}")
        print()

### Display raw records for each record set by their `@id`
This step uses the `records(record_set=<id>)` method from `mlcroissant`. Replace `<id>` with actual @id from above.

In [ ]:
# Display first records from each record set using their @id
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nSample records for record set {rs_id}:")
    try:
        recs = dataset.records(record_set=rs_id)
        for i, rec in enumerate(recs):
            print(rec)
            if i >= 3:
                break
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

## 3. Data Extraction
Load data from available record sets into DataFrames. Use record set and field `@id` values.

In [ ]:
# Build a list of record set @ids for extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for {rs_id} with shape {df.shape}")
            print(f"Columns: {df.columns.tolist()}")
            print("Sample:")
            print(df.head())
        else:
            print(f"No records found for {rs_id}.")
    except Exception as e:
        print(f"Error loading records for {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All columns are referenced by their `@id`.

In [ ]:
# For demonstration, pick the first populated record set
first_record_set_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        first_record_set_id = rs_id
        break
if first_record_set_id is None:
    print("No populated record sets found for EDA.")
else:
    print(f"Selected record set @id for EDA: {first_record_set_id}")
    df = dataframes[first_record_set_id]

    # Find a numeric field by checking dtypes, and referencing fields by their @id
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a possible group field (categorical column with < 20 unique values)
        group_fields = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < 20]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found in this record set.")

## 5. Visualization
Visualize data distributions and relationships between fields referenced by their `@id`.

In [ ]:
# Quick visualization: histogram and boxplot for numeric fields
import matplotlib.pyplot as plt
import seaborn as sns

if first_record_set_id and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.show()

    if group_fields:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields or record sets available for visualization.")

## 6. Conclusion
In this notebook, we explored the RNA sequence features dataset referenced via its Croissant schema. Using `mlcroissant`, we loaded its metadata, examined available record sets and fields (with references by their `@id`), performed basic data extraction and analysis, and visualized the data.

Key steps include:
- Loading and reviewing dataset metadata
- Identifying and loading record set data by `@id`
- Filtering and normalizing data using referenced fields
- Visualizing data distributions by field `@id`

Further domain-specific analyses, such as biological feature annotation or cross-platform comparison, can be pursued with more detailed exploration.
